# Boolean Expressions: Solutions and Test Evidence

Reference: [CSP 3.05 Boolean Expressions](https://pages.opencodingsociety.com/python/boolean/2027).

Each task below has a prediction, executable Python, saved output, and a brief explanation.
`=` assigns a value in Python; `==` compares values. `and` requires every condition,
`or` permits either match, and `not` reverses a Boolean.

## Popcorn Hack 1: Check One SFI Part Record

**Prediction:** The supplied record passes all four field checks. Replacing its effective
date with spaces makes the date check and the complete-record result `False`.

**Explanation:** Each check produces a Boolean. Combining them with `and` requires all
four checks to pass. A complete record can still have a duplicate spec; uniqueness is
tested separately in Hack 2.

In [1]:
# CODE_RUNNER: Popcorn Hack 1 - Check each required SFI field. | python_runtime: browser

part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}

valid_categories = ["Auto Racing", "Drag Racing"]

has_product_name = isinstance(part.get("product_name"), str) and bool(part["product_name"].strip())
valid_category = part.get("category") in valid_categories
has_spec_number = isinstance(part.get("spec_number"), str) and bool(part["spec_number"].strip())
has_effective_date = isinstance(part.get("effective_date"), str) and bool(part["effective_date"].strip())

record_is_complete = has_product_name and valid_category and has_spec_number and has_effective_date

print("Product name present:", has_product_name)
print("Category accepted:", valid_category)
print("Spec number present:", has_spec_number)
print("Effective date present:", has_effective_date)
print("Record complete:", record_is_complete)

# A whitespace-only field should fail even though the string is not empty.
part["effective_date"] = "   "
has_effective_date = bool(part["effective_date"].strip())
record_is_complete = has_product_name and valid_category and has_spec_number and has_effective_date
print("After clearing the effective date:", record_is_complete)


Product name present: True
Category accepted: True
Spec number present: True
Effective date present: True
Record complete: True
After clearing the effective date: False


### Observed output

```text
Product name present: True
Category accepted: True
Spec number present: True
Effective date present: True
Record complete: True
After clearing the effective date: False
```


## Popcorn Hack 2: Backend Create Rule

**Prediction:**

| Case | Expected decision | Reason |
| --- | --- | --- |
| Original spec `1.1` | REJECTED | Already stored |
| Change spec to `1.2` | ACCEPTED | Complete, accepted category, new spec |
| Change category to Street Car | REJECTED | Category is not accepted |
| Remove effective date | REJECTED | Required field is missing |

**Explanation:** `and` combines the field checks. A spec is new only when it is present
and `not in` the stored spec list. The first run rejects the supplied duplicate record.

In [2]:
# CODE_RUNNER: Popcorn Hack 2 - Test accepted and rejected create decisions. | python_runtime: browser

part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}

valid_categories = ["Auto Racing", "Drag Racing"]
existing_spec_numbers = ["1.1", "2.1"]


def can_create_sfi_record(record, accepted_categories, existing_specs):
    """Create only complete records with an accepted category and a new spec."""
    product_name = record.get("product_name", "")
    category = record.get("category", "")
    spec_number = record.get("spec_number", "")
    effective_date = record.get("effective_date", "")

    has_product_name = isinstance(product_name, str) and bool(product_name.strip())
    valid_category = isinstance(category, str) and category.strip() in accepted_categories
    has_spec_number = isinstance(spec_number, str) and bool(spec_number.strip())
    has_effective_date = isinstance(effective_date, str) and bool(effective_date.strip())
    spec_is_new = has_spec_number and spec_number.strip() not in existing_specs

    return (
        has_product_name
        and valid_category
        and has_spec_number
        and has_effective_date
        and spec_is_new
    )


# The original 1.1 spec is already stored; 1.2 is a valid new spec.
test_cases = [
    ("Duplicate spec", part),
    ("New spec", {**part, "spec_number": "1.2"}),
    ("Invalid category", {**part, "spec_number": "1.2", "category": "Street Car"}),
    ("Missing effective date", {**part, "spec_number": "1.2", "effective_date": ""}),
]

for label, candidate in test_cases:
    accepted = can_create_sfi_record(candidate, valid_categories, existing_spec_numbers)
    print(f"{label}: {'ACCEPTED' if accepted else 'REJECTED'}")


Duplicate spec: REJECTED
New spec: ACCEPTED
Invalid category: REJECTED
Missing effective date: REJECTED


### Observed output

```text
Duplicate spec: REJECTED
New spec: ACCEPTED
Invalid category: REJECTED
Missing effective date: REJECTED
```


## Popcorn Hack 3: SFI Search Filter

**Prediction:**

| Input | Expected matches |
| --- | --- |
| Exact spec `1.2` | Multiple Disc Clutch Assemblies |
| Product query `FLYWHEEL` | Specs `1.1` and `2.1` |
| Query `no match` | None |
| Change the matching record to Street Car | None |

**Explanation:** `(product_matches or spec_matches) and category_is_valid` keeps the
category rule attached to both search options. Parentheses prevent an invalid-category
product-name match from bypassing the filter. Spec matching is exact; product matching
is case-insensitive. An empty query returns no results.

In [3]:
# CODE_RUNNER: Popcorn Hack 3 - Match valid records by name or exact spec. | python_runtime: browser

query = "1.2"

records = [
    {
        "product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    }
]

valid_categories = ["Auto Racing", "Drag Racing"]


def search_sfi_records(records, query, accepted_categories):
    """Match a product name OR exact spec, AND require an accepted category."""
    if not isinstance(query, str):
        raise ValueError("Search query must be a string.")
    search_text = query.strip().casefold()
    if not search_text:
        return []

    matches = []
    for record in records:
        product_name = record.get("product_name", "")
        spec_number = record.get("spec_number", "")
        category = record.get("category", "")

        product_matches = isinstance(product_name, str) and search_text in product_name.casefold()
        spec_matches = isinstance(spec_number, str) and search_text == spec_number.strip().casefold()
        category_is_valid = isinstance(category, str) and category.strip() in accepted_categories

        if (product_matches or spec_matches) and category_is_valid:
            matches.append(record)
    return matches


for search_query in [query, "FLYWHEEL", "no match"]:
    results = search_sfi_records(records, search_query, valid_categories)
    print(f"Query {search_query!r}: {len(results)} match(es)")
    for record in results:
        print(f"  {record['spec_number']}: {record['product_name']}")

# An exact spec match must still be rejected when its category is not accepted.
changed_records = [{**records[1], "category": "Street Car"}]
filtered = search_sfi_records(changed_records, "1.2", valid_categories)
print("After changing the matching record to Street Car:", len(filtered), "match(es)")


Query '1.2': 1 match(es)
  1.2: Multiple Disc Clutch Assemblies
Query 'FLYWHEEL': 2 match(es)
  1.1: Replacement Flywheels and Clutch Assemblies
  2.1: Racing Flywheel Record
Query 'no match': 0 match(es)
After changing the matching record to Street Car: 0 match(es)


### Observed output

```text
Query '1.2': 1 match(es)
  1.2: Multiple Disc Clutch Assemblies
Query 'FLYWHEEL': 2 match(es)
  1.1: Replacement Flywheels and Clutch Assemblies
  2.1: Racing Flywheel Record
Query 'no match': 0 match(es)
After changing the matching record to Street Car: 0 match(es)
```


## In-class MCQ Result

**Actual result: Not yet recorded.** Complete the quiz on the linked lesson and paste
its exact result line here. This notebook does not claim a quiz score.

## College Board Pseudocode Check

The assignment arrow stores a value. In pseudocode, `=` compares values; in Python,
the corresponding comparison is `==`.

```text
fieldsOK ← namePresent AND categoryAllowed AND specPresent AND datePresent
accept ← fieldsOK AND NOT alreadyStored
IF (accept)
{
    DISPLAY("ACCEPTED")
}
ELSE
{
    DISPLAY("REJECTED")
}
```

With all fields valid, spec `1.2` is new: `True AND NOT False` predicts ACCEPTED.
For stored spec `1.1`, `True AND NOT True` predicts REJECTED. The Python tests below
produce those same decisions.

## Independent Homework: SFI Backend Validator

**Prediction:**

| Record | Expected result | Reason |
| --- | --- | --- |
| 1: Multiple Disc Clutch Assemblies, spec `1.2` | ACCEPTED | All five checks pass |
| 2: Replacement Flywheels | REJECTED | Street Car category |
| 3: Existing Flywheel Record | REJECTED | Duplicate spec `1.1` |
| 4: Whitespace-only product name | REJECTED | Missing name |
| 5: Whitespace-only spec | REJECTED | Missing spec |
| 6: Missing effective date | REJECTED | Missing date |

**Explanation:** The validator returns one Boolean from five named conditions.
Short-circuiting protects calls to `strip()` from non-string values. The caller uses
that Boolean to print the decision. This cell defines its own inputs and function,
so it runs independently of the popcorn hacks.

In [4]:
# CODE_RUNNER: Validate each candidate and print ACCEPTED or REJECTED. | python_runtime: browser
# Homework Hack - SFI Backend Validator

existing_spec_numbers = ["1.1", "2.1"]
valid_categories = ["Auto Racing", "Drag Racing"]

test_records = [
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.2",
        "effective_date": "Feb. 9, 2006"
    },
    {
        "product_name": "Replacement Flywheels",
        "category": "Street Car",
        "spec_number": "3.1",
        "effective_date": "Jan. 1, 2026"
    },
    {
        "product_name": "Existing Flywheel Record",
        "category": "Drag Racing",
        "spec_number": "1.1",
        "effective_date": "Jan. 1, 2026"
    }
]

# Extra cases exercise the remaining required-field checks.
test_records.extend([
    {
        "product_name": "   ",
        "category": "Auto Racing",
        "spec_number": "4.1",
        "effective_date": "Jan. 1, 2026"
    },
    {
        "product_name": "Missing Specification",
        "category": "Drag Racing",
        "spec_number": "   ",
        "effective_date": "Jan. 1, 2026"
    },
    {
        "product_name": "Missing Effective Date",
        "category": "Auto Racing",
        "spec_number": "4.2",
        "effective_date": ""
    }
])


def validate_sfi_record(record, accepted_categories, existing_specs):
    """Return whether a candidate passes all five required-record checks.

    Parameters:
        record (dict): Candidate with product_name, category, spec_number,
            and effective_date string fields. Missing fields fail validation.
        accepted_categories (collection[str]): Permitted racing category names.
        existing_specs (collection[str]): Spec numbers already in the backend.
            Category names and existing specs should be stored without padding.

    Returns:
        bool: True when all fields are present, the category is accepted,
            and the spec number is new; False otherwise. Empty or whitespace-
            only strings and non-string field values fail validation.
    """
    product_name = record.get("product_name", "")
    category = record.get("category", "")
    spec_number = record.get("spec_number", "")
    effective_date = record.get("effective_date", "")

    # strip() prevents a field containing only spaces from counting as filled.
    has_product_name = isinstance(product_name, str) and bool(product_name.strip())
    category_is_accepted = isinstance(category, str) and category.strip() in accepted_categories
    has_spec_number = isinstance(spec_number, str) and bool(spec_number.strip())
    has_effective_date = isinstance(effective_date, str) and bool(effective_date.strip())

    # Short-circuiting skips strip() on invalid values. NOT IN rejects duplicates.
    spec_is_new = has_spec_number and spec_number.strip() not in existing_specs

    # AND means one failed requirement is enough to reject the entire record.
    return (
        has_product_name
        and category_is_accepted
        and has_spec_number
        and has_effective_date
        and spec_is_new
    )


for index, candidate in enumerate(test_records, start=1):
    accepted = validate_sfi_record(candidate, valid_categories, existing_spec_numbers)
    status = "ACCEPTED" if accepted else "REJECTED"
    product = str(candidate.get("product_name") or "").strip() or "(missing product name)"
    spec = str(candidate.get("spec_number") or "").strip() or "(missing spec number)"
    print(f"Record {index}: {status} | {product} | Spec: {spec}")


Record 1: ACCEPTED | Multiple Disc Clutch Assemblies | Spec: 1.2
Record 2: REJECTED | Replacement Flywheels | Spec: 3.1
Record 3: REJECTED | Existing Flywheel Record | Spec: 1.1
Record 4: REJECTED | (missing product name) | Spec: 4.1
Record 5: REJECTED | Missing Specification | Spec: (missing spec number)
Record 6: REJECTED | Missing Effective Date | Spec: 4.2


### Observed output

```text
Record 1: ACCEPTED | Multiple Disc Clutch Assemblies | Spec: 1.2
Record 2: REJECTED | Replacement Flywheels | Spec: 3.1
Record 3: REJECTED | Existing Flywheel Record | Spec: 1.1
Record 4: REJECTED | (missing product name) | Spec: 4.1
Record 5: REJECTED | Missing Specification | Spec: (missing spec number)
Record 6: REJECTED | Missing Effective Date | Spec: 4.2
```


## Submission Notes

After publishing, open [the notebook page]({{ '/python/boolean-hw' | relative_url }}).
Submit only the Python from the independent homework cell in the lesson's code form;
keep the notebook link and test evidence in Notes.

```text
Lesson: CSP 3.05 Boolean Expressions
Notebook: <paste the published URL after deployment>
MCQ 3.05: <paste your actual quiz result>
Popcorn Hacks 1–3: predictions, code, and saved output included
Homework tested: one accepted record and five rejected records
```

Remaining submission items: record the actual MCQ result, publish this notebook,
and replace the URL placeholder with the deployed page address.